## Optuna practise

In [1]:
# import necessary linararies
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#load the pima indian dataset from sklearn
#NBote:-Scikit learn built in "load_diabetes"is a regression model
#we will load the actual diabetets from an external source

import pandas as pd
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']
df=pd.read_csv(url,names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [3]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [5]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-09-21 23:39:16,283] A new study created in memory with name: no-name-9450e852-9e76-4960-861f-ae4e33118442
[I 2025-09-21 23:39:16,772] Trial 0 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 107, 'max_depth': 20}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-21 23:39:17,229] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 91, 'max_depth': 16}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-21 23:39:17,497] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 64, 'max_depth': 11}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-21 23:39:18,011] Trial 3 finished with value: 0.7746741154562383 and parameters: {'n_estimators': 130, 'max_depth': 12}. Best is trial 0 with value: 0.7746741154562384.
[I 2025-09-21 23:39:18,929] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 171, 'max_depth': 13}. Best is trial 0 with value: 0.774674

In [6]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 196, 'max_depth': 17}


In [7]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


## sampler in Optuna

## for random serach CV

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [9]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-09-21 23:39:49,315] A new study created in memory with name: no-name-6228be2e-85f8-459b-9bb1-bfca30930721


[I 2025-09-21 23:39:49,606] Trial 0 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 67, 'max_depth': 17}. Best is trial 0 with value: 0.7783985102420857.
[I 2025-09-21 23:39:49,963] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 85, 'max_depth': 11}. Best is trial 0 with value: 0.7783985102420857.
[I 2025-09-21 23:39:50,545] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 122, 'max_depth': 11}. Best is trial 0 with value: 0.7783985102420857.
[I 2025-09-21 23:39:50,865] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 53, 'max_depth': 14}. Best is trial 0 with value: 0.7783985102420857.
[I 2025-09-21 23:39:51,644] Trial 4 finished with value: 0.7523277467411545 and parameters: {'n_estimators': 186, 'max_depth': 6}. Best is trial 0 with value: 0.7783985102420857.
[I 2025-09-21 23:39:52,436] Trial 5 finished with value: 0.7653631284916201 and parameters: {'n_estimato

In [10]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7858472998137803
Best hyperparameters: {'n_estimators': 121, 'max_depth': 15}


##  For Grid Search CV

In [11]:
# we explctly define searh_space in gridsearchcv
search_space={
    'n_estimators':[50,100,150,200],
    'max_depth':[5,10,15,20]
}

In [12]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-09-21 23:40:20,226] A new study created in memory with name: no-name-3821b892-a042-4faf-bd85-79f583c8f0f1
[I 2025-09-21 23:40:20,654] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-09-21 23:40:21,394] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-09-21 23:40:21,645] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-09-21 23:40:22,151] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-09-21 23:40:22,687] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [13]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


## Optuna Visualization

In [14]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [15]:
#Optimization HIstory
plot_optimization_history(study).show()

In [16]:
# parrallel coodrinate chord
plot_parallel_coordinate(study).show()

In [17]:
#3 sl,ice plot
plot_slice(study).show()

In [18]:
#4 plot contour Plot
plot_contour(study).show()

In [19]:
#5 Hyparameter Importance
plot_param_importances(study).show()

In [21]:
#best thing inoptine
# define by run
# Dynamic Search Species

### Optimizing Multiple ML Models

In [22]:
#library
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC

In [29]:

#define the obbjection functon for the optima
def objective(trial):
    #choose the algorithm to tune
    classifier_name=trial.suggest_categorical('classifier',['SVM','RandomForest','GradientBoosting'])
    
    if classifier_name == "SVM":
        #SVM Hyperparameter
        c=trial.suggest_float('C',0.1,100,log=True)
        kernel=trial.suggest_categorical('kernel',['linear','rbf','poly','sigmoid'])
        gamma=trial.suggest_categorical('gamma',['scale','auto'])
        model=SVC(C=c,kernel=kernel,gamma=gamma,random_state=42)
        
    elif classifier_name == "RandomForest":
        #random forst hypermeters
        n_estimators=trial.suggest_int('n_estimators',50,300)
        max_depth=trial.suggest_int('max_depth',3,20)
        min_samples_split=trial.suggest_int('min_samples_split',2,10)
        min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10)
        bootstrap=trial.suggest_categorical('bootstrap',[True,False])
        
        model=RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )
    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        
    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [30]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100) 
#  default tpe sample

[I 2025-09-22 00:10:59,721] A new study created in memory with name: no-name-7ffbe6de-962c-4e70-a2f9-798aeb8b8c73
[I 2025-09-22 00:10:59,761] Trial 0 finished with value: 0.7858472998137803 and parameters: {'classifier': 'SVM', 'C': 0.9079480897210014, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 0 with value: 0.7858472998137803.
[I 2025-09-22 00:11:00,332] Trial 1 finished with value: 0.756052141527002 and parameters: {'classifier': 'RandomForest', 'n_estimators': 150, 'max_depth': 11, 'min_samples_split': 9, 'min_samples_leaf': 9, 'bootstrap': True}. Best is trial 0 with value: 0.7858472998137803.
[I 2025-09-22 00:11:00,352] Trial 2 finished with value: 0.7169459962756052 and parameters: {'classifier': 'SVM', 'C': 0.1642886687461777, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7858472998137803.
[I 2025-09-22 00:11:00,376] Trial 3 finished with value: 0.7113594040968342 and parameters: {'classifier': 'SVM', 'C': 0.11715919939395705, 'kernel': 'poly', 'gam

In [31]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.15316515437136982, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [33]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.785847,2025-09-22 00:10:59.723798,2025-09-22 00:10:59.761202,0 days 00:00:00.037404,0.907948,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.756052,2025-09-22 00:10:59.762567,2025-09-22 00:11:00.332579,0 days 00:00:00.570012,NaN,True,RandomForest,NaN,NaN,NaN,11.0,9.0,9.0,150.0,COMPLETE
2,2,0.716946,2025-09-22 00:11:00.333637,2025-09-22 00:11:00.352798,0 days 00:00:00.019161,0.164289,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.711359,2025-09-22 00:11:00.353797,2025-09-22 00:11:00.376631,0 days 00:00:00.022834,0.117159,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.765363,2025-09-22 00:11:00.377631,2025-09-22 00:11:02.697440,0 days 00:00:02.319809,NaN,NaN,GradientBoosting,NaN,NaN,0.010881,17.0,8.0,8.0,285.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.759777,2025-09-22 00:11:25.596213,2025-09-22 00:11:25.628968,0 days 00:00:00.032755,0.281178,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.757914,2025-09-22 00:11:25.630017,2025-09-22 00:11:26.097848,0 days 00:00:00.467831,NaN,False,RandomForest,NaN,NaN,NaN,8.0,2.0,9.0,113.0,COMPLETE
97,97,0.761639,2025-09-22 00:11:26.098848,2025-09-22 00:11:26.953711,0 days 00:00:00.854863,NaN,NaN,GradientBoosting,NaN,NaN,0.017310,4.0,4.0,5.0,165.0,COMPLETE
98,98,0.783985,2025-09-22 00:11:26.953711,2025-09-22 00:11:26.988026,0 days 00:00:00.034315,1.455134,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [35]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
GradientBoosting    11
RandomForest        10
Name: count, dtype: int64

In [36]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()


params_classifier
GradientBoosting    0.750127
RandomForest        0.762011
SVM                 0.776513
Name: value, dtype: float64